# Pisto-GPT-64m — Arabic Pretraining (from scratch, 6h)
Trains the 68M Arabic base model from scratch on Kaggle (2×T4).

**Before running**: enable **Internet** (Settings → turn on Internet) and select **GPU** (2xT4).
The run is capped at 6h by `config/train.json` (`max_hours`). It uses both GPUs via DataParallel.
After it finishes, download `weights/pretrain_best.pt` and upload it to your Kaggle dataset
(`etoooooo9/pisto-weights`) so the SFT notebook can use the new base.

In [ ]:
!pip install -q torch datasets tokenizers

In [ ]:
!rm -rf pg && git clone -q https://github.com/BayanDrp/pisto-gpt-64m.git pg
%cd pg
!git log --oneline -1
!cat config/train.json


In [ ]:
import sys
sys.path.insert(0, 'llm')
from tokenizer import ByteTokenizer
tk = ByteTokenizer('config/bpe_tokenizer.json')
print('vocab_size =', tk.vocab_size)
assert tk.vocab_size == 8192, 'BAD TOKENIZER — wrong bpe_tokenizer.json'
print('tokenizer OK (correct Arabic BPE, vocab 8192)')


In [ ]:
import os, glob
# Train FROM SCRATCH: remove any stale pretrain checkpoint so we do not resume an old base.
for p in glob.glob('weights/pretrain_best.pt') + glob.glob('weights/pretrain_last.pt'):
    os.remove(p); print('removed', p)
os.makedirs('weights', exist_ok=True)
print('fresh start — no resume')


In [ ]:
import os
# Paste your HuggingFace token (role=Read) to lift dataset download rate limits.
# Get it at huggingface.co -> Settings -> Access Tokens. Do NOT commit the real token.
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


In [ ]:
%cd pg
!PRETRAIN_CONFIG=pretrain_en.json python -u training/pretrain.py

In [ ]:
!ls -la weights/
print()
print('Done. Download weights/pretrain_best.pt and upload it to your Kaggle dataset')
print('(etoooooo9/pisto-weights) so the SFT notebook (kaggle_train.ipynb) uses the new base.')
